# Attribution Modeling Dashboard

Loads the sample click/sale/spend data, runs all four attribution models, and scores channel efficiency (CPC, CAC, ROI).

Swap the CSVs in `data/sample/` for your own data with the same schema to point this at a real business.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

%load_ext jupysql

engine = create_engine("sqlite:///attribution.db")
%sql engine --alias attribution

In [ ]:
import sys
sys.path.append('..')

from utils.parse_channels import parse_channels
from utils.datetime_fix import normalize_datetimes

ad_channels = pd.read_csv('../data/sample/ad_channels_sample.csv')
clicks = pd.read_csv('../data/sample/clicks_sample.csv')
sales = pd.read_csv('../data/sample/sales_sample.csv')
spend = pd.read_csv('../data/sample/spend_sample.csv')

ad_channels = parse_channels(ad_channels)

normalized = normalize_datetimes(
    {'clicks': clicks, 'sales': sales},
    {'clicks': ['click_datetime'], 'sales': ['sale_datetime']}
)
clicks, sales = normalized['clicks'], normalized['sales']

ad_channels.head()

In [ ]:
%sql --persist-replace ad_channels
%sql --persist-replace clicks
%sql --persist-replace sales
%sql --persist-replace spend

## Attribution Models

Run each model against the same data to compare how credit shifts across channels.

In [ ]:
%sql --file ../models/first_touch.sql

In [ ]:
%sql --file ../models/last_touch.sql

In [ ]:
%sql --file ../models/linear.sql

In [ ]:
%sql --file ../models/time_decay.sql

## Channel Efficiency

CPC, CAC, and journey-shape queries. These only need clicks/sales/spend, so they run directly against the sample data.

In [ ]:
%sql --file ../queries/cpc_by_channel.sql

In [ ]:
%sql --file ../queries/cac_by_partner.sql

In [ ]:
%sql --file ../queries/unattributed_sales.sql

`channels_before_convert.sql` was written against a `locks` table (Carvana's term for a locked-in purchase decision). In this generalized schema a lock is just the sale event, so we expose it as a view over `sales` before running the query unmodified.

In [ ]:
%sql CREATE VIEW IF NOT EXISTS locks AS SELECT sale_id AS lock_id, user_id, sale_datetime AS lock_datetime FROM sales

In [ ]:
%sql --file ../queries/channels_before_convert.sql

## ROI by Channel

`roi_by_channel.sql` needs a `sale_profit` table built by `utils/profit_calc.py`, which in turn needs a product/vehicle table (bodystyle, avg_margin, etc.) joined onto sales. That product-level table isn't part of `data/sample/` — the sample only covers clicks, sales, spend, and channels.

To run this section: supply your own `products_df` (one row per product/vehicle with `avg_margin` and `bodystyle`, matched to sales via `make`/`model` or your own keys), then:

```python
from utils.profit_calc import run_profit_pipeline
sale_profit = run_profit_pipeline(sales, products_df)
%sql --persist-replace sale_profit
%sql --file ../queries/roi_by_channel.sql
```